In [ ]:
from pathlib import Path
import pandas as pd
import cv2
from sklearn.metrics import classification_report
import numpy as np

Выявление наличия ооцита из line_detection.py

In [ ]:
def preprocess(gray, target_mean=0.85, target_std=0.155):
    gray_f = gray.astype(np.float32) / 255.0
    mean_val = np.mean(gray_f)
    std_val = np.std(gray_f)

    alpha = np.clip(target_std / (std_val + 1e-6), 0.6, 2.0)
    beta = np.clip((target_mean - mean_val * alpha) * 255, -80, 80)

    adjusted = cv2.convertScaleAbs(gray, alpha=alpha, beta=beta)

    mean_after = np.mean(adjusted.astype(np.float32) / 255.0)
    gamma = np.clip(1.0 + (target_mean - mean_after) * 1.5, 0.6, 1.4)
    img_gamma = np.power(adjusted.astype(np.float32) / 255.0, gamma)
    corrected = np.clip(img_gamma * 255, 0, 255).astype(np.uint8)

    blurred = cv2.GaussianBlur(corrected, (5, 5), 0)
    return blurred


def detect_oocyte(gray):
    """
    Возвращает:
        True — если ооцит найден
        False — если не найден
    """
    g = preprocess(gray)

    # бинаризация
    bin_img = cv2.adaptiveThreshold(
        g, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        35, 2
    )
    bin_inv = 255 - bin_img

    # поиск контуров
    contours, _ = cv2.findContours(
        bin_inv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    if not contours:
        return False

    h, w = gray.shape[:2]
    best_score = -1

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < h * w * 0.01:       # маленькие отбросить
            continue

        perim = cv2.arcLength(cnt, True)
        if perim == 0 or len(cnt) < 5:
            continue

        ellipse = cv2.fitEllipse(cnt)
        (cx, cy), (ma, MA), angle = ellipse

        # форма — не очень вытянутая
        ratio = max(ma, MA) / min(ma, MA)
        if ratio > 2.5:
            continue

        # центр должен быть примерно в центре кадра
        if not (w * 0.2 < cx < w * 0.8 and h * 0.2 < cy < h * 0.8):
            continue

        circ = 4 * np.pi * area / (perim * perim)
        score = circ * np.sqrt(area)

        best_score = max(best_score, score)

    return best_score > 0


In [3]:
base_dir = Path(r"D:\diplom\dataset_detect")
csv_path = base_dir / "oocyte_detect.csv"

In [4]:
df = pd.read_csv(csv_path)

In [7]:
y_true = []
y_pred = []

for idx, row in df.iterrows():
    name = str(row["name"])
    label = int(row["oocyte"])

    img_path = base_dir / f"{name}.png"

    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"[WARN] Не удалось открыть {img_path}")
        continue

    pred = detect_oocyte(img)
    y_true.append(label)
    y_pred.append(1 if pred else 0)

In [8]:
print("=== Classification Report ===")
print(classification_report(y_true, y_pred, digits=4))

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.6368    0.9412    0.7596       136
           1     0.9843    0.8733    0.9255       576

    accuracy                         0.8862       712
   macro avg     0.8106    0.9072    0.8426       712
weighted avg     0.9180    0.8862    0.8938       712



Выявление наличия ооцита из line_r_theta_processed.py

In [9]:
def preprocess_frame(gray):
    img = gray.astype(np.float32) / 255.0
    img = np.clip(img, 0, 1)

    blur1 = cv2.GaussianBlur(img, (31, 31), 0)
    blur2 = cv2.GaussianBlur(img, (63, 63), 0)
    blur3 = cv2.GaussianBlur(img, (127, 127), 0)
    local_mean = (blur1 + blur2 + blur3) / 3.0

    c1, c2 = 2.6, 0.46
    q = np.tan(np.pi / 2 * c1 * (local_mean + c2))
    q = np.nan_to_num(q, nan=0.0, posinf=1.0, neginf=0.0)
    L_corr = np.sin(np.pi / 2 * np.clip(q * img, 0, 1))

    win_size = 15
    mean_local = cv2.blur(L_corr, (win_size, win_size))
    sq_mean_local = cv2.blur(L_corr ** 2, (win_size, win_size))
    variance = np.maximum(sq_mean_local - mean_local ** 2, 1e-6)

    sigma_min = np.percentile(variance, 1)
    k = 0.8
    p = k * np.log(np.clip(variance / sigma_min, 1.0, None))

    local_avg = cv2.GaussianBlur(L_corr, (15, 15), 0)
    enhanced = local_avg + p * (L_corr - local_avg)

    enhanced = np.clip(enhanced, 0, 1)
    target_brightness = 0.65
    mean_intensity = np.mean(enhanced)
    if mean_intensity > 1e-3:
        scale = np.clip(target_brightness / mean_intensity, 0.8, 1.5)
        enhanced *= scale

    denoised = cv2.bilateralFilter((enhanced * 255).astype(np.uint8),
                                   d=7, sigmaColor=15, sigmaSpace=7)
    denoised = denoised.astype(np.float32) / 255.0

    blur_sharp = cv2.GaussianBlur(denoised, (0, 0), 1.2)
    sharpened = cv2.addWeighted(denoised, 1.5, blur_sharp, -0.5, 0)
    sharpened = np.clip(sharpened, 0, 1)

    enhanced_uint8 = (sharpened * 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(enhanced_uint8, (3, 3), 0)

    return blurred


def find_oocyte_contour(gray_img):
    h, w = gray_img.shape[:2]
    denoised = cv2.bilateralFilter(gray_img, d=7, sigmaColor=30, sigmaSpace=15)
    denoised = cv2.medianBlur(denoised, 3)

    alpha, beta = 1.5, 8
    enhanced = cv2.convertScaleAbs(denoised, alpha=alpha, beta=beta)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(enhanced)

    otsu_t, _ = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    adjusted_t = int(otsu_t * 0.85)
    _, binary = cv2.threshold(enhanced, adjusted_t, 255, cv2.THRESH_BINARY_INV)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    best_score = -1
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < h * w * 0.01:
            continue

        perim = cv2.arcLength(cnt, True)
        if perim == 0 or len(cnt) < 5:
            continue

        circularity = 4 * np.pi * area / (perim ** 2)
        score = circularity * np.sqrt(area)

        if score > best_score:
            M = cv2.moments(cnt)
            if M["m00"] == 0:
                continue
            cx = M["m10"] / M["m00"]
            cy = M["m01"] / M["m00"]
            if not (w * 0.2 < cx < w * 0.8 and h * 0.2 < cy < h * 0.8):
                continue

            best_score = score

    return None if best_score < 0 else True


def detect_oocyte_r_theta_processed(gray):
    processed = preprocess_frame(gray)
    found = find_oocyte_contour(processed)
    return found is not None

In [11]:
y_true = []
y_pred = []

for idx, row in df.iterrows():
    name = str(row["name"])
    label = int(row["oocyte"])

    img_path = base_dir / f"{name}.png"

    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"[WARN] cannot read {img_path}")
        continue

    pred = detect_oocyte_r_theta_processed(img)
    y_true.append(label)
    y_pred.append(1 if pred else 0)

print("=== Classification Report ===")
print(classification_report(y_true, y_pred, digits=4))


=== Classification Report ===
              precision    recall  f1-score   support

           0     0.5769    0.7721    0.6604       136
           1     0.9415    0.8663    0.9024       576

    accuracy                         0.8483       712
   macro avg     0.7592    0.8192    0.7814       712
weighted avg     0.8719    0.8483    0.8561       712



Выявление наличия ооцита из line_r_theta.py

In [12]:
def preprocess_frame(gray):
    alpha = 1.6
    beta = 10
    adjusted = cv2.convertScaleAbs(gray, alpha=alpha, beta=beta)

    gray_f = adjusted.astype(np.float32) / 255.0
    gamma = 0.8
    corrected = np.power(gray_f, gamma)
    adjusted = np.clip(corrected * 255, 0, 255).astype(np.uint8)

    blurred = cv2.GaussianBlur(adjusted, (3, 3), 0)
    return blurred


def find_oocyte_contour(gray_img):
    _, binary = cv2.threshold(
        gray_img, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    best_contour = None
    best_score = 0

    for contour in contours:
        if len(contour) < 5:
            continue
        area = cv2.contourArea(contour)
        perimeter = cv2.arcLength(contour, True)
        if perimeter == 0:
            continue
        circularity = 4 * np.pi * area / (perimeter ** 2)

        if area > 3000 and 0.6 < circularity <= 1.2:
            if circularity > best_score:
                best_score = circularity
                best_contour = contour

    if best_contour is None:
        for contour in contours:
            if len(contour) < 5:
                continue
            area = cv2.contourArea(contour)
            perimeter = cv2.arcLength(contour, True)
            if perimeter == 0:
                continue
            circularity = 4 * np.pi * area / (perimeter ** 2)

            if area > 2000 and 0.4 < circularity <= 1.3:
                if circularity > best_score:
                    best_score = circularity
                    best_contour = contour

    return best_contour


def detect_oocyte_r_theta(gray):
    """
    Возвращает:
        True  — ооцит найден
        False — ооцит отсутствует
    """
    processed = preprocess_frame(gray)
    contour = find_oocyte_contour(processed)
    return contour is not None


In [14]:
y_true = []
y_pred = []

for _, row in df.iterrows():
    name = str(row["name"])        # БЕЗ расширения
    label = int(row["oocyte"])

    img_path = base_dir / f"{name}.png"

    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"[WARN] Не удалось открыть {img_path}")
        continue

    pred = detect_oocyte_r_theta(img)
    y_true.append(label)
    y_pred.append(1 if pred else 0)


print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, digits=4))



=== Classification Report ===
              precision    recall  f1-score   support

           0     0.2749    0.4265    0.3343       136
           1     0.8443    0.7344    0.7855       576

    accuracy                         0.6756       712
   macro avg     0.5596    0.5804    0.5599       712
weighted avg     0.7355    0.6756    0.6993       712

